# Predicting the Winner of the 2026 FIFA World Cup

https://www.kaggle.com/datasets/cashncarry/fifaworldranking/data
\
https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017/data

# Inroduction

With the 2026 FIFA World Cup now just a few months away, anticipation for the tournament is growing rapidly worldwide. In British Columbia, competition feels even more exciting for residents this year, as Vancouver is one of the official host cities and will host seven matches at BC Place. As the world's biggest sporting event approaches, predicting which of the 48 nations will win the tournament becomes a compelling data science question. By using historical World Cup results, international match data, and team rankings, it is possible to identify patterns that may help determine the strongest contenders for the 2026 title. 

### Maybe like a few photos here? 

The relevance of the World Cup in relation to Vancouver has motivated us to develop a prdeictive model to tend to the question; **Are we able to predict the nations with the largest chances to win the 2026 Word Cup with previous matches, and historical Word Cup results? Moreover, if we are able to make these predictions, what model would allow us to make this prediction?** 

With our two data sets, we have variables related to different countries FIFA rankings, as well as previous match history.

**The first dataset(https://www.kaggle.com/datasets/cashncarry/fifaworldranking/data) consists of the following:**

* **country_full - Countries Full Name**

* **country_abrv - Countries Abbreviated Name**

* **rank - Current Country Rank**

* **total_points — Current Total Points**

* **previous_points — Total Points in Last Rating**

* **rank_change — How Rank Has Changed Since The Last Publication**

* **confederation — FIFA Confederations**

* **rank_date — Date of Rating Calculation**

**The second dataset(https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017/data) consists of the follwing columns:**

* **date - Date of The Match**

* **away_team - Name of The Away Team**

* **home_score - Full-time Home Team Score Including Extra Time (Not Including Penalty-Shootouts)**

* **away_score - Full-Time Away Team Score Including Extra Time (Not Including Penalty-Shootouts)**

* **tournament - Name of The Tournament**

* **city - Name of The City/Town/Administrative Unit Where The Match Was Played**

* **country - Name of The Cuntry The Match Was Played In**

* **neutral - TRUE/FALSE Column Indicating Whether The Match Was Played At a Neutral Venue**

In [24]:
import altair as alt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

Loading the results dataset

In [7]:
results_data = pd.read_csv('https://raw.githubusercontent.com/sayebm/Group_Project/refs/heads/main/results.csv')
results_data

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False
...,...,...,...,...,...,...,...,...,...
49066,2026-01-18,Bolivia,Panama,1,1,Friendly,Tarija,Bolivia,False
49067,2026-01-18,Grenada,Jamaica,0,1,Friendly,St. George's,Grenada,False
49068,2026-01-22,Panama,Mexico,0,1,Friendly,Panama City,Panama,False
49069,2026-01-25,Bolivia,Mexico,0,1,Friendly,Santa Cruz,Bolivia,False


Sorting the results data to only consider obervations(matches) from 2015 onwards

In [8]:
# Only considering matches starting after 2015
results_data['date'] = pd.to_datetime(results_data['date'])
results_filtered = results_data[results_data['date'] >= '2015-01-01']
results_filtered

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
38380,2015-01-04,Bahrain,Jordan,1,0,Friendly,Ballarat,Australia,True
38381,2015-01-04,Iran,Iraq,1,0,Friendly,Wollongong,Australia,True
38382,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,Parramatta,Australia,True
38383,2015-01-04,South Africa,Zambia,1,0,Friendly,Johannesburg,South Africa,False
38384,2015-01-05,China PR,Oman,4,1,Friendly,Penrith,Australia,True
...,...,...,...,...,...,...,...,...,...
49066,2026-01-18,Bolivia,Panama,1,1,Friendly,Tarija,Bolivia,False
49067,2026-01-18,Grenada,Jamaica,0,1,Friendly,St. George's,Grenada,False
49068,2026-01-22,Panama,Mexico,0,1,Friendly,Panama City,Panama,False
49069,2026-01-25,Bolivia,Mexico,0,1,Friendly,Santa Cruz,Bolivia,False


Loading the rankings dataset and Converting the rank_date variable into a 'date' for python

In [9]:
rankings_data = pd.read_csv('https://raw.githubusercontent.com/sayebm/Group_Project/refs/heads/main/fifa_ranking-2024-06-20.csv')
rankings_data["year"] = pd.to_datetime(rankings_data["rank_date"]).dt.year
rankings_data


,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date,year
0,140.0,Brunei Darussalam,BRU,2.00,0.00,140,AFC,1992-12-31,1992
1,33.0,Portugal,POR,38.00,0.00,33,UEFA,1992-12-31,1992
2,32.0,Zambia,ZAM,38.00,0.00,32,CAF,1992-12-31,1992
3,31.0,Greece,GRE,38.00,0.00,31,UEFA,1992-12-31,1992
4,30.0,Algeria,ALG,39.00,0.00,30,CAF,1992-12-31,1992
...,...,...,...,...,...,...,...,...,...
67467,137.0,Kuwait,KUW,1098.42,1085.46,-2,AFC,2024-06-20,2024
67468,136.0,Lithuania,LTU,1100.66,1095.23,-1,UEFA,2024-06-20,2024
67469,135.0,Malaysia,MAS,1107.58,1094.54,-3,AFC,2024-06-20,2024
67470,133.0,Solomon Islands,SOL,1111.02,1111.02,1,OFC,2024-06-20,2024


Filter the rankings data to only consider rankings from 2015 onwards and for each country to only have 1 rankings per year

In [10]:
# Only considering rankings starting after 2015 and only have 1 observation per year per team
rankings_filtered = (
    rankings_data[rankings_data['year'] >= 2015]
    .groupby(['country_full', 'year'])
    .last()
    .reset_index()
)
rankings_filtered

,country_full,year,rank,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,Afghanistan,2015,150.0,AFG,194.00,168.00,-6,AFC,2015-12-03
1,Afghanistan,2016,146.0,AFG,189.00,189.00,-1,AFC,2016-12-22
2,Afghanistan,2017,148.0,AFG,181.00,181.00,1,AFC,2017-12-21
3,Afghanistan,2018,147.0,AFG,1068.00,1068.00,0,AFC,2018-12-20
4,Afghanistan,2019,149.0,AFG,1052.00,1052.00,0,AFC,2019-12-19
...,...,...,...,...,...,...,...,...,...
2101,Zimbabwe,2020,108.0,ZIM,1181.00,1181.00,0,CAF,2020-12-10
2102,Zimbabwe,2021,121.0,ZIM,1138.44,1138.44,0,CAF,2021-12-23
2103,Zimbabwe,2022,125.0,ZIM,1138.56,1138.56,0,CAF,2022-12-22
2104,Zimbabwe,2023,124.0,ZIM,1144.56,1144.56,0,CAF,2023-12-21


Some countries are spelled differently in the datasets, so this makes them the same

In [11]:
name_mapping = {
    'USA':            'United States',
    'Korea Republic': 'South Korea',
    'Congo DR':       'DR Congo',
    "Cote d'Ivoire":  'Ivory Coast',
    'IR Iran':        'Iran',
}
rankings_data['country_full'] = rankings_data['country_full'].replace(name_mapping)
rankings_filtered['country_full'] = rankings_filtered['country_full'].replace(name_mapping)

Teams that have already qualified for the 2026 world cup

In [12]:
world_cup_2026_teams = [
    # Host nations (auto-qualified)
    "United States", "Canada", "Mexico",
    # UEFA
    "Germany", "Portugal", "Spain", "France", "England", "Netherlands",
    "Belgium", "Austria", "Turkey", "Poland", "Serbia",
    # CONMEBOL
    "Argentina", "Colombia", "Uruguay", "Brazil", "Ecuador", "Paraguay",
    # CONCACAF
    "Panama", "Costa Rica", "Honduras", "Jamaica",
    # CAF
    "Morocco", "Egypt", "Senegal", "South Africa", "DR Congo",
    "Ivory Coast", "Nigeria", "Algeria", "Tunisia",
    # AFC
    "Iran", "South Korea", "Japan", "Saudi Arabia", "Australia",
    "Uzbekistan", "Jordan", "Iraq",
    # OFC
    "New Zealand",
]

Filter to only include matches playd between two teams that have qualified for the 2026 world cup

In [13]:
# Only includes teams currently qualified for the wolrd cup
results_wc2026 = results_filtered[
    results_filtered['home_team'].isin(world_cup_2026_teams) &
    results_filtered['away_team'].isin(world_cup_2026_teams)
]
results_wc2026

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
38381,2015-01-04,Iran,Iraq,1,0,Friendly,Wollongong,Australia,True
38382,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,Parramatta,Australia,True
38395,2015-01-11,Nigeria,Ivory Coast,0,1,Friendly,Abu Dhabi,United Arab Emirates,True
38396,2015-01-11,Tunisia,Algeria,1,1,Friendly,Radès,Tunisia,False
38399,2015-01-12,Jordan,Iraq,0,1,AFC Asian Cup,Brisbane,Australia,True
...,...,...,...,...,...,...,...,...,...
49062,2026-01-14,Senegal,Egypt,1,0,African Cup of Nations,Tangier,Morocco,True
49063,2026-01-14,Morocco,Nigeria,0,0,African Cup of Nations,Rabat,Morocco,False
49064,2026-01-17,Egypt,Nigeria,0,0,African Cup of Nations,Casablanca,Morocco,True
49065,2026-01-18,Morocco,Senegal,0,1,African Cup of Nations,Rabat,Morocco,False


Filter to only include rankings for teams that are qualified for the 2026 world cup

In [14]:
# Only includes teams currently qualified for the wolrd cup
rankings_wc2026 = rankings_filtered[
    rankings_filtered['country_full'].isin(world_cup_2026_teams)
].sort_values(by=['rank_date', 'rank'], ascending=[False, True])
rankings_wc2026

,country_full,year,rank,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
89,Argentina,2024,1.0,ARG,1860.14,1858.00,0,CONMEBOL,2024-06-20
717,France,2024,2.0,FRA,1837.47,1840.59,0,UEFA,2024-06-20
199,Belgium,2024,3.0,BEL,1797.98,1795.23,0,UEFA,2024-06-20
279,Brazil,2024,4.0,BRA,1791.85,1788.65,-1,CONMEBOL,2024-06-20
627,England,2024,5.0,ENG,1787.88,1794.90,1,UEFA,2024-06-20
...,...,...,...,...,...,...,...,...,...
967,Jordan,2015,87.0,JOR,399.00,411.00,5,AFC,2015-12-03
360,Canada,2015,88.0,CAN,388.00,335.00,-14,CONCACAF,2015-12-03
917,Iraq,2015,89.0,IRQ,381.00,392.00,2,AFC,2015-12-03
847,Honduras,2015,101.0,HON,338.00,359.00,6,CONCACAF,2015-12-03


Adding nations FIFA ranking for each match played

In [15]:
# So the 2 datasets can correlate the countries rank at the time of a match with each other

results_wc2026['year'] = results_wc2026['date'].dt.year
rankings_lookup = rankings_wc2026[['country_full', 'year', 'rank']]


results_wc2026 = (
    results_wc2026.merge(
    rankings_lookup.rename(columns={'country_full': 'home_team', 'rank': 'home_ranking'})
    ).merge(
    rankings_lookup.rename(columns={'country_full': 'away_team', 'rank': 'away_ranking'})
    )
)
results_wc2026 = results_wc2026[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'neutral', 'home_ranking', 'away_ranking']]
results_wc2026


/var/folders/pb/9jlgn7sn1qnc1ndptsdnjrs00000gn/T/ipykernel_7517/3029627807.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_wc2026['year'] = results_wc2026['date'].dt.year


,date,home_team,away_team,home_score,away_score,tournament,neutral,home_ranking,away_ranking
0,2015-01-04,Iran,Iraq,1,0,Friendly,True,45.0,89.0
1,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,True,51.0,80.0
2,2015-01-11,Tunisia,Algeria,1,1,Friendly,False,40.0,28.0
3,2015-01-12,Jordan,Iraq,0,1,AFC Asian Cup,True,87.0,89.0
4,2015-01-16,Iraq,Japan,0,1,AFC Asian Cup,True,89.0,53.0
...,...,...,...,...,...,...,...,...,...
1018,2024-11-18,United States,Jamaica,4,2,CONCACAF Nations League,False,11.0,53.0
1019,2024-11-19,Mexico,Honduras,4,0,CONCACAF Nations League,False,15.0,78.0
1020,2024-11-19,Colombia,Ecuador,0,1,FIFA World Cup qualification,False,12.0,30.0
1021,2024-11-19,Brazil,Uruguay,1,1,FIFA World Cup qualification,False,4.0,14.0


### Functions that sorts the 'Tournament' column in the DataFrame

In [16]:
'''
def get_tournament_types(t = 'tournament'):
    
    #Returns the different types of values in the tournaments column
    
    return results_data[t].drop_duplicates().sort_values()

tournaments = get_tournament_types()

tournaments
'''

"\ndef get_tournament_types(t = 'tournament'):\n\n    #Returns the different types of values in the tournaments column\n\n    return results_data[t].drop_duplicates().sort_values()\n\ntournaments = get_tournament_types()\n\ntournaments\n"

### Displays how many times each unique tournament type is mentioned in the data under the tournaments column

In [17]:
#results_wc2026['tournament'].value_counts()

### Assigns a numerical weight to each tournament type (1 = most important, and 0 = meaningless)

I will not include the 'tournament' that only occured 2-3 times in our data.

We can manipulate these values in case our data in the end does not seem reasonable

In [18]:
'''
tournament_weights = {
    'FIFA World Cup': 1.00,
    'Copa América' : 0.85,
    'UEFA Euro' : 0.85,
    'AFC Asian Cup': 0.85,
    'African Cup of Nations': 0.85,
    'Gold Cup' : 0.80,
    'Confedertions Cup': 0.75, # Lower because it was discontinuied in 2019 and the final tournamnet was in 2017
    'CONCACAF Nations League': 0.75,
    'UEFA Nations League': 0.75,
    'Superclásico de las Américas': 0.65,
    'FIFA World Cup qualification': 0.65,
    'EAFF Championship': 0.65,
    'Arab Cup': 0.60,
    'UEFA Euro qaulification': 0.55,
    'Copa América qualification': 0.55,
    'WAFF Championship': 0.55,
    'FIFA Series': 0.55,
    'Gulf Cup': 0.55,
    'African Cup of Nations qualification': 0.50,
    'UNCAF Cup': 0.50,
    'COSAFA Cup': 0.50,
    'CAFA Nations Cup': 0.5,
    'Kirin Challenge Cup': 0.35,
    'Kirin Cup': 0.35,
    'Soccer Ashes': 0.30,
    'Friendly': 0.20,
}

results_wc2026_copy = results_wc2026.copy()                             #Copies filtered data 
results_wc2026_copy['tournaments_weight'] = (                           #Creates a new column called 'tournament_weight'
    results_wc2026_copy['tournament'].map(tournament_weights).fillna(0.30) #Assigns 0.30 to any tournament that was not mentioned 
)

results_wc2026_copy[['tournament', 'tournament_weight']                     #displays the new columns,removes duplicates,                     
    ].drop_duplicates().sort_values('tournament_weight', ascending = False) # and sorts them from highest to lowest weight
'''

"\ntournament_weights = {\n    'FIFA World Cup': 1.00,\n    'Copa América' : 0.85,\n    'UEFA Euro' : 0.85,\n    'AFC Asian Cup': 0.85,\n    'African Cup of Nations': 0.85,\n    'Gold Cup' : 0.80,\n    'Confedertions Cup': 0.75, # Lower because it was discontinuied in 2019 and the final tournamnet was in 2017\n    'CONCACAF Nations League': 0.75,\n    'UEFA Nations League': 0.75,\n    'Superclásico de las Américas': 0.65,\n    'FIFA World Cup qualification': 0.65,\n    'EAFF Championship': 0.65,\n    'Arab Cup': 0.60,\n    'UEFA Euro qaulification': 0.55,\n    'Copa América qualification': 0.55,\n    'WAFF Championship': 0.55,\n    'FIFA Series': 0.55,\n    'Gulf Cup': 0.55,\n    'African Cup of Nations qualification': 0.50,\n    'UNCAF Cup': 0.50,\n    'COSAFA Cup': 0.50,\n    'CAFA Nations Cup': 0.5,\n    'Kirin Challenge Cup': 0.35,\n    'Kirin Cup': 0.35,\n    'Soccer Ashes': 0.30,\n    'Friendly': 0.20,\n}\n\nresults_wc2026_copy = results_wc2026.copy()                             

if home team wins it would be 1 and if it loses 0

In [19]:
results_wc2026["home_win"] = (results_wc2026["home_score"] > results_wc2026["away_score"]).astype(int)
results_wc2026["draw"] = results_wc2026["home_score"] == results_wc2026["away_score"]


model_data = results_wc2026.loc[~results_wc2026["draw"]].copy() #only rows that matches are not a draw


model_data["rank_diff"] = model_data["away_ranking"] - model_data["home_ranking"]#comapre strength


model_data = model_data.dropna(subset=["home_ranking", "away_ranking", "neutral"])
model_data

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_ranking,away_ranking,home_win,draw,rank_diff
0,2015-01-04,Iran,Iraq,1,0,Friendly,True,45.0,89.0,1,False,44.0
1,2015-01-04,South Korea,Saudi Arabia,2,0,Friendly,True,51.0,80.0,1,False,29.0
3,2015-01-12,Jordan,Iraq,0,1,AFC Asian Cup,True,87.0,89.0,0,False,2.0
4,2015-01-16,Iraq,Japan,0,1,AFC Asian Cup,True,89.0,53.0,0,False,-36.0
5,2015-01-17,Australia,South Korea,0,1,AFC Asian Cup,False,57.0,51.0,0,False,-6.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1016,2024-11-15,Uruguay,Colombia,3,2,FIFA World Cup qualification,False,14.0,12.0,1,False,-2.0
1018,2024-11-18,United States,Jamaica,4,2,CONCACAF Nations League,False,11.0,53.0,1,False,42.0
1019,2024-11-19,Mexico,Honduras,4,0,CONCACAF Nations League,False,15.0,78.0,1,False,63.0
1020,2024-11-19,Colombia,Ecuador,0,1,FIFA World Cup qualification,False,12.0,30.0,0,False,18.0


Get the columns we want to train on and what our target is 

In [20]:
X = model_data [[
    "home_ranking",
    "away_ranking",
    "rank_diff",
    "neutral",
]]

y= model_data["home_win"]
X

,home_ranking,away_ranking,rank_diff,neutral
0,45.0,89.0,44.0,True
1,51.0,80.0,29.0,True
3,87.0,89.0,2.0,True
4,89.0,53.0,-36.0,True
5,57.0,51.0,-6.0,False
...,...,...,...,...
1016,14.0,12.0,-2.0,False
1018,11.0,53.0,42.0,False
1019,15.0,78.0,63.0,False
1020,12.0,30.0,18.0,False


Split into training and testing set

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=123,
    stratify=y,
)

Scale the numeric values and keep the binary value as is 

In [22]:
numeric_features = ["home_ranking", "away_ranking", "rank_diff"]
binary_features = ["neutral"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        ("bin", "passthrough", binary_features),  
    ],
    remainder="drop",
)

### Tests Different K values for KNN using cross-validation and picks the one with the highest accuracy after preprocessing the data.

In [25]:
knn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier()),
])

param_grid = {
    "model__n_neighbors": range(1, 21, 2),
}

knn_grid = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)

knn_grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__n_neighbors': range(1, 21, 2)}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and